# Part I: Removing a Node from a Linked List

In class we built a simple singly linked list that could add nodes and traverse them. Your job is to extend it with a `remove` method that deletes a node by its string value.

## Specification

Add a method `remove(self, value: str) -> bool` to the linked list class from class. It should:

- Find the **first** node whose value equals `value` and unlink it from the list.
- Return `True` if a node was removed, `False` if no matching node was found.
- Leave the list in a valid state afterward — meaning `head` still points to the correct first node (or `None` if the list is now empty), and traversing from `head` visits exactly the remaining nodes in their original order.

The list should not contain a "hole" or a dangling reference after removal. If you remove the only node in the list, the list should become empty.

## Classes to use

Use [`LinkedList`](https://github.com/lgreco/COMP-271-SP26/blob/bd3467676acbaf694165a154b761b9d9717a4fa5/week09/linked_list.py#L4) and [`Node`](https://github.com/lgreco/COMP-271-SP26/blob/bd3467676acbaf694165a154b761b9d9717a4fa5/week09/node.py) from week 09.

## Cases to Think Through

Before you write a single line of code, sketch what has to happen in each of these situations. They are not all the same, and the one that trips people up is usually the first one.

1. **Removing the head node.** There is no "previous" node to rewire. What has to change?
2. **Removing a middle node.** You need the node *before* the target so you can splice around it. How do you keep track of the previous node as you walk the list?
3. **Removing the tail node.** Does your middle-node logic already handle this, or is it a separate case?
4. **The value isn't in the list.** You walk off the end. Return `False`.
5. **The list is empty to begin with.** Return `False` immediately — don't crash.

A common clean approach is to carry two references as you walk: `current` and `previous`. When `current.value == value`, you either update `self.head` (if `previous is None`) or set `previous.next = current.next`. Then return `True`.

## Tips

- Don't reach for recursion. An iterative two-pointer walk is shorter, clearer, and won't blow the stack on long lists.
- If you find yourself writing `if self.head.value == value` as a special case at the top of the method, that's fine — it's often the cleanest way to handle the head. Just make sure the rest of the method doesn't duplicate work you've already done.
- After you get it working, read your method out loud. If you can't explain in one sentence what each line does, simplify.

## Stretch

Add a second method `remove_all(self, value: str) -> int` that removes *every* node matching `value` and returns the count of nodes removed. Be careful: once you unlink a node, the "next node to check" is the one your previous pointer now points at — don't accidentally skip it.

## Part II: `WeirdoBST` — A Binary Search Tree in an Array

### Background: Two Different Trees, One Familiar Trick

You've already seen how a **complete binary tree** can be stored in a plain Python list without any node objects, pointers, or references. The trick is purely arithmetic: for a node stored at index `i`,

- its left child lives at index `2*i + 1`,
- its right child lives at index `2*i + 2`,
- its parent lives at index `(i - 1) // 2`.

This works beautifully for structures like a **max-heap**, where we deliberately keep the tree complete (every level full, except possibly the last, which fills left-to-right). Because the tree is complete, an array of size exactly $n$ holds all $n$ items with zero wasted slots, and the height stays at $\Theta(\log_2 n)$. Life is good.

Now let's break that.

### The Twist: A *Binary Search Tree* in an Array

A **binary search tree (BST)** is shaped by the data, not by a promise of completeness. If strings arrive in the order `"A"`, `"B"`, `"C"`, the resulting BST is a sad little right-leaning stick:

```text
A
 \
  B
   \
    C
```

There are still only $n = 3$ items, but the tree has height 3. If we insist on storing this BST using the same index arithmetic as a heap — root at 0, left child at `2*i+1`, right child at `2*i+2` — then `"C"` ends up at index 6, and every unused slot between the root and `"C"` has to exist and hold `None`:

```text
index:  0     1      2     3     4     5      6
value: [A,    None,  B,    None, None, None,  C ]
```

So an array of length 7 stores just 3 items. **That is the point of this exercise.** You're going to feel the inefficiency in your bones, and that feeling is the lesson: the clean index math that makes heaps so elegant becomes wasteful the moment the tree is allowed to be unbalanced. In the worst case (sorted input), $n$ items can require an array of length $2^n - 1$. Embrace it.

### Your Task

Implement a class `WeirdoBST` that stores string values in a BST, where the tree is represented internally as a Python list using the heap-style index arithmetic described above. The class must support:

- `insert(self, value: str) -> None` — insert a string, preserving the BST ordering (left subtree < node < right subtree). Assume no duplicates for simplicity, or decide on a sensible policy and document it.
- `search(self, value: str) -> bool` — return whether the string is present.
- `__len__(self)` — return $n$, the number of actual strings stored (not the length of the underlying array).
- `__str__(self)` — something useful for printing.


### Implementation Tips

1. **Start with the backing store.** In `__init__`, make `self._underlying: list[str | None] = []`. Don't pre-allocate a size — grow on demand.

2. **Growing the array.** When you need to write at index `i` and `i >= len(self._underlying)`, extend the list with `None` values until it's long enough. Hint: no need to do any elaborate math like exponentiation of logarithms here.

3. **Insertion is a walk, not a recursion on objects.** Start at index `i = 0`. If `self._underlying[i]` is `None`, you've found the spot — write the value there (growing first if needed). Otherwise compare the incoming value to `self._underlying[i]` and step to `2*i + 1` or `2*i + 2`. Repeat. The loop terminates when you write into a `None` slot.

4. **Search is the same walk, minus the writing.** Walk from index 0, comparing and branching left or right, until you either find the value or step off the end of the array (or land on a `None`, which also means "not here" — why?).

5. **`__len__` is not `len(self._underlying)`.** It's the count of non-`None` entries. You can compute it on demand with a helper function, or maintain a running count in an attribute. Either is fine; the running count is faster but easier to get wrong.

6. **Test with a pathological case.** Insert `"A"`, `"B"`, `"C"`, `"D"` in order and print the backing array. You should see the array balloon while only four slots are actually occupied. Then insert `"Z"`, `"M"`, `"F"` in an order that produces a bushier tree and watch the array fill in more densely. Contrast the two — that contrast *is* the takeaway.

7. **Don't try to rebalance.** Do not attempt to improve efficiency. The whole point is to let the inefficiency show.

8. **Think about what this would cost in the worst case.** After implementing it, ask yourself: if I insert $n$ strings in sorted order, how long is `self._underlying` at the end? Write the answer in a comment at the top of your file. (Hint: it's not linear in $n$.)

### Reflection Questions (to include with your submission)

- Compare the space used by `WeirdoBST` to a pointer-based BST storing the same data (meaning TreeNode objects). Under what input distributions is the gap small? When is it catastrophic?
- Why does the heap's array (complete binary tree) representation avoid this problem entirely?


## What to submit

A file called `week13a.py` with you code for Part I and a file called `week13b.py` with your code for Part II, uploaded on Sakai. If your work is done in a Jupyter Notebook, make the notebook shareable and provide its link as the sole line of your `week13a.py` and `week13b.py` files respectively.